[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_5_Grouping_a_Series.ipynb)

# 06.5: Grouping a Series

Data often contains values that belong together. If you have one row for each NBA championship, you may want to know how many times each team has won, or when each team first won. You could write a mask for each team, but there are 23 teams in this data. This short notebook introduces `.groupby()`, which does the same work for every group at once.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

This series has one item for each NBA champion from 1947 to 2020. The label is the team and the value is the year. Team names repeat in the index, as you saw is allowed in notebook 06.1.

In [ ]:
import pandas as pd

wlist = ['Los Angeles Lakers', 'Toronto Raptors', 'Golden State Warriors',
  'Golden State Warriors', 'Cleveland Cavaliers', 'Golden State Warriors',
  'San Antonio Spurs', 'Miami Heat', 'Miami Heat', 'Dallas Mavericks',
  'Los Angeles Lakers', 'Los Angeles Lakers', 'Boston Celtics',
  'San Antonio Spurs', 'Miami Heat', 'San Antonio Spurs', 'Detroit Pistons',
  'San Antonio Spurs', 'Los Angeles Lakers', 'Los Angeles Lakers',
  'Los Angeles Lakers', 'San Antonio Spurs', 'Chicago Bulls', 'Chicago Bulls',
  'Chicago Bulls', 'Houston Rockets', 'Houston Rockets', 'Chicago Bulls',
  'Chicago Bulls', 'Chicago Bulls', 'Detroit Pistons', 'Detroit Pistons',
  'Los Angeles Lakers', 'Los Angeles Lakers', 'Boston Celtics',
  'Los Angeles Lakers', 'Boston Celtics', 'Philadelphia 76ers',
  'Los Angeles Lakers', 'Boston Celtics', 'Los Angeles Lakers',
  'Seattle Supersonics', 'Washington Bullets', 'Portland Trail Blazers',
  'Boston Celtics', 'Golden State Warriors', 'Boston Celtics','New York Knicks',
  'Los Angeles Lakers', 'Milwaukee Bucks', 'New York Knicks', 'Boston Celtics',
  'Boston Celtics', 'Philadelphia 76ers', 'Boston Celtics', 'Boston Celtics',
  'Boston Celtics', 'Boston Celtics', 'Boston Celtics', 'Boston Celtics',
  'Boston Celtics', 'Boston Celtics', 'St. Louis Hawks', 'Boston Celtics',
  'Philadelphia Warriors', 'Syracuse Nationals', 'Minneapolis Lakers',
  'Minneapolis Lakers', 'Minneapolis Lakers', 'Rochester Royals',
  'Minneapolis Lakers', 'Minneapolis Lakers', 'Baltimore Bullets',
  'Philadelphia Warriors']


nba_series = pd.Series(range(2020, 1946, -1), index=wlist)
nba_series.head(3)

The Los Angeles Lakers won in 2020, the Toronto Raptors in 2019 and the Golden State Warriors in 2018.

## Using `.groupby()`

Grouping takes two steps. First, `.groupby()` makes an object that knows how to split the items into groups. We tell it to group by the index, so items with the same team name go together.

In [ ]:
nba_grouped = nba_series.groupby(by=nba_series.index)
nba_grouped

The output is not a result yet. It only says that this is a groupby object, because we have not said what to calculate. The second step is to call a method on it. `.count()` counts the items in each group, which here is the number of championships.

In [ ]:
nba_grouped.count()

The result is a series with one item per team, with the team names in alphabetical order. The Baltimore Bullets have 1. Because it is an ordinary series, you can sort it to answer the question of who won the most.

In [ ]:
nba_grouped.count().sort_values(ascending=False).head(3)

The Boston Celtics lead with 17, followed by the Los Angeles Lakers with 12 and the Chicago Bulls with 6. The other statistics from 06.3 work the same way, including `.mean()`, `.median()`, `.sum()`, `.min()` and `.max()`. What year was each team's first championship? That is the smallest year in each group, and the most recent is the largest.

In [ ]:
print(nba_grouped.min().head(3))
print(nba_grouped.max().head(3))

The Boston Celtics first won in 1957 and most recently in 2008, in this data. Since these results are series with the same labels, you can subtract them using what you learned in 06.2, which gives the time between each team's first and last win.

In [ ]:
(nba_grouped.max() - nba_grouped.min()).sort_values(ascending=False).head(3)

The Celtics have the longest span at 51 years, then the Lakers at 48 and the Golden State Warriors at 43.

## Looking at one group

`.get_group()` returns the items for one group.

In [ ]:
nba_grouped.get_group("Chicago Bulls")

The Bulls won in 1991, 1992, 1993, 1996, 1997 and 1998. You could get the same items with `nba_series.loc["Chicago Bulls"]`, since the label is repeated.

## Grouping by something you calculated

The Minneapolis Lakers became the Los Angeles Lakers, so counting by the full name splits one franchise into two. If we keep only the last word of each team name, which is the `.str` method from 06.4 applied to the index, those teams combine.

In [ ]:
team_only = nba_series.index.str.split(" ").str[-1]
nba_series.groupby(by=team_only).count().sort_values(ascending=False).head(3)

The Lakers now have 17, which ties them with the Celtics, and the Bulls still have 6. This changes the answer to who has won the most, so it is worth asking whether the groups you have are the ones you want. Grouping by the last word is not perfect, though, since it also combines any two different teams with the same nickname.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

store_list = ["Bloomington", "Normal", "Peoria", "Normal", "Bloomington", "Peoria",
              "Normal", "Bloomington", "Normal", "Peoria"]
sale_list = [120, 95, 160, 110, 150, 180, 70, 90, 125, 155]
sales = pd.Series(sale_list, index=store_list)
sales.head(3)

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a groupby object named `by_store` that groups `sales` by its index.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
print("Problem 1: correct." if "by_store" in globals() and "groupby" in str(type(by_store)).lower() else "Problem 1: not yet. Call .groupby() with by=sales.index, and do not add a calculation yet.")

### Problem 2
Create a series named `store_counts` with the number of sales for each store.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "store_counts", sales.groupby(by=sales.index).count(), "Call .count() on your groupby object.")

### Problem 3
Create a series named `store_means` with the mean sale for each store, ordered from highest to lowest.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "store_means", sales.groupby(by=sales.index).mean().sort_values(ascending=False), "Call .mean() and then sort the result from high to low.")

### Problem 4
Create a series named `normal_sales` with only the sales for the group `'Normal'`, using `.get_group()`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "normal_sales", sales.loc["Normal"], "Pass the name of the group to .get_group().")

### Problem 5
Create a series named `store_spread` with the difference between the largest and the smallest sale for each store.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
grouped_check = sales.groupby(by=sales.index)
check(5, "store_spread", grouped_check.max() - grouped_check.min(), "Subtract .min() from .max().")

## What's next

That finishes module 06. You can now create a series, look up and filter its values, summarize it, work with text in it and summarize groups. Real datasets have many columns, though, and a question usually involves more than one of them. Module 07 starts with notebook 07.1, which introduces the DataFrame, a table made of series that share an index.